<!-- notebook-header -->
# Regressao Estatistica: Teoria e Diagnostico

**Modulo:** 01 - Estatistica  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** OLS, diagnostico de residuos, multicolinearidade, regularizacao e learning curves.


# Regressao Estatistica: da Teoria ao Diagnostico

## Indice

1. [Regressao Linear Simples](#1)
2. [Regressao Linear Multipla](#2)
3. [Diagnostico de Residuos](#3)
4. [Multicolinearidade e VIF](#4)
5. [Regularizacao: Ridge, Lasso, ElasticNet](#5)
6. [Selecao de Modelos com Validacao Cruzada](#6)
7. [Regressao Polinomial e Overfitting](#7)
8. [Learning Curves: Diagnostico de Vies/Variancia](#8)
9. [Exercicios Praticos](#9)
10. [Erros Comuns e Armadilhas](#10)
11. [Resumo e Conexoes](#11)

## Pre-requisitos e Fio Narrativo

| Conceito | Notebook | Importancia |
|----------|----------|-------------|
| Algebra Linear (matrizes, inversas) | 0_3 | Alta |
| Calculo (derivadas, otimizacao) | 0_4 | Media |
| Estatistica Inferencial (testes, ICs) | 1_2 | Fundamental |
| Estatistica Bayesiana (MAP, regularizacao) | 1_3 | Alta |
| Estatistica Descritiva (correlacao) | 1_1 | Fundamental |

**Tempo estimado:** 10-12 horas

**Fio narrativo:** Nos notebooks anteriores, voce aprendeu a DESCREVER dados (1_1), TESTAR hipoteses sobre eles (1_2), e ATUALIZAR crencas com novos dados (1_3). Agora voce vai combinar tudo isso para PREVER valores continuos. Regressao linear e o modelo mais fundamental de ML: simples o suficiente para ter solucao analitica, complexo o suficiente para ensinar todos os conceitos de modelagem (overfitting, regularizacao, diagnostico, validacao).

**Objetivos de aprendizado:**
- Compreender regressao linear simples e multipla (OLS)
- Diagnosticar pressupostos e problemas nos residuos
- Detectar e lidar com multicolinearidade (VIF)
- Usar regularizacao (Ridge, Lasso, ElasticNet)
- Selecionar modelos com validacao cruzada
- Diagnosticar vies/variancia com learning curves

## Por que Regressao em ML?

Regressao linear e a FUNDACAO de quase todo ML supervisionado:

- **Modelo base**: E o primeiro modelo que voce deve testar em qualquer problema de regressao (baseline)
- **Regressao logistica**: Classificacao binaria e regressao linear + funcao sigmoide
- **Redes neurais**: Cada neuronio e uma regressao linear seguida de uma nao-linearidade
- **Regularizacao**: Ridge (L2) e Lasso (L1) se aplicam a TAREFA DO ALUNOS os modelos lineares e a maioria dos nao-lineares
- **Interpretabilidade**: Coeficientes de regressao dao explicabilidade que modelos complexos nao dao
- **Diagnostico**: As tecnicas de diagnostico de residuos se aplicam a qualquer modelo (nao so regressao)
- **Feature engineering**: Polynomial features, interacoes, transformacoes - todos sao extensoes de regressao
- **Gradient descent**: A otimizacao de regressao linear e o caso mais simples de gradient descent (0_8)

## 1. Regressao Linear Simples

### Analogia: A Linha que Melhor Descreve Seus Dados

Imagine dados de altura vs peso. Claramente tem relacao: pessoas mais altas tendem a pesar mais. Regressao linear simples encontra UMA UNICA RETA:

y = beta_0 + beta_1 * x

Onde beta_0 (intercept) e onde a reta cruza o eixo y, e beta_1 (slope) e a inclinacao.

Como encontra a melhor reta? Minimiza a soma dos erros ao quadrado (OLS - Ordinary Least Squares). Por que quadrado? Porque penaliza outliers, tem solucao analitica, e tem propriedades estatisticas boas (BLUE - Best Linear Unbiased Estimator).

O R^2 (coeficiente de determinacao) responde: "Quanto da variacao em y e explicada por x?" Se R^2 = 0.7, significa 70% da variacao e explicada.

### Definicao Formal

- Modelo: y = beta_0 + beta_1 * x + epsilon, onde epsilon ~ Normal(0, sigma^2)
- Solucao OLS: beta_1 = Cov(x,y) / Var(x), beta_0 = y_barra - beta_1 * x_barra
- R^2 = 1 - SS_res / SS_tot = 1 - SUM(y_i - y_hat_i)^2 / SUM(y_i - y_barra)^2

### Por que em ML?

Regressao linear simples ensina o conceito de FUNCAO DE CUSTO (MSE) e OTIMIZACAO (encontrar parametros que minimizam o custo). Esses dois conceitos sao a base de TAREFA DO ALUNO ML supervisionado, de redes neurais a gradient boosting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import shapiro, jarque_bera
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')

print('=== REGRESSAO LINEAR ===')
print('\nModelo: y = beta_0 + beta_1*x_1 + beta_2*x_2 + ... + beta_p*x_p + eps')
print('Objetivo: Minimizar Sum((y_i - y_hat_i)^2) (soma de erros ao quadrado)')
print('Solucao fechada: beta = (X^T X)^-1 X^T y')


In [ ]:
# Dados de exemplo
np.random.seed(42)
X = np.random.randn(50) * 10 + 50
y = 2 * X + np.random.randn(50) * 15 + 30

print('\n=== REGRESSÃO LINEAR SIMPLES ===' )
print(f'\nDados: {len(X)} observações')

# Modelo
X_reshape = X.reshape(-1, 1)
model = LinearRegression()
model.fit(X_reshape, y)

y_pred = model.predict(X_reshape)

beta_0 = model.intercept_
beta_1 = model.coef_[0]

print(f'\nEquação: y = {beta_0:.2f} + {beta_1:.2f} × x')

# Métricas
ss_res = np.sum((y - y_pred)**2)  # Soma de erros ao quadrado
ss_tot = np.sum((y - y.mean())**2)  # Soma total de variância
r2 = 1 - (ss_res / ss_tot)
rse = np.sqrt(ss_res / (len(y) - 2))  # Residual Standard Error

print(f'\nMétricas:')
print(f'  R² = {r2:.3f} ({r2*100:.1f}% da variância explicada)')
print(f'  RSE (Residual Standard Error) = {rse:.2f}')
print(f'  RMSE = {np.sqrt(mean_squared_error(y, y_pred)):.2f}')

# Visualização
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(X, y, alpha=0.6, s=50)
ax.plot(X, y_pred, 'r-', linewidth=2, label=f'y = {beta_0:.1f} + {beta_1:.2f}x (R² = {r2:.3f})')
ax.set_xlabel('X')
ax.set_ylabel('y')
ax.set_title('Regressão Linear Simples')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### O que observar

- A reta vermelha minimiza a distancia vertical de todos os pontos a reta
- O R^2 indica a proporcao da variancia explicada pelo modelo
- RSE (Residual Standard Error) estima o desvio padrao dos residuos
- RMSE e RSE sao similares mas RSE ajusta pelos graus de liberdade

### O que concluir

- R^2 alto nao garante que o modelo e bom (pode ser overfitting ou relacao nao-linear)
- R^2 NUNCA diminui ao adicionar mais variaveis (use R^2 ajustado para comparar modelos)
- Sempre visualize os dados: o famoso "Quarteto de Anscombe" mostra 4 datasets com mesmo R^2 mas padroes completamente diferentes
- A equacao y = beta_0 + beta_1*x so e valida no RANGE dos dados (nao extrapole!)

### Conexao com outros notebooks

- **0_4 (Calculo)**: Minimizar MSE e um problema de otimizacao; derivar e igualar a zero da a solucao analitica
- **0_3 (Algebra Linear)**: A solucao matricial beta = (X'X)^-1 X'y usa inversao de matrizes
- **1_1 (Descritiva)**: Correlacao (r) e a raiz quadrada de R^2 em regressao simples

## 2. Regressao Linear Multipla

### Analogia: Do Plano ao Hiperplano

Regressao simples usa UMA variavel para prever y. Multipla usa VARIAS: y = beta_0 + beta_1*x_1 + beta_2*x_2 + ... + beta_p*x_p + epsilon

Cada coeficiente beta_j representa a mudanca em y quando x_j aumenta 1 unidade, MANTENDO todas as outras variaveis constantes (ceteris paribus).

### Definicao Formal

Em forma matricial: y = X * beta + epsilon

Solucao OLS: beta = (X'X)^-1 X'y

Pressupostos: E[epsilon] = 0, Var(epsilon) = sigma^2 * I (homocedasticidade), epsilon ~ Normal (para inferencia)

### Por que em ML?

A maioria dos problemas reais tem multiplas features. Regressao multipla ensina como combinar features, interpretar coeficientes parciais, e detectar multicolinearidade - conceitos que se aplicam a qualquer modelo linear generalizado.

In [ ]:
# Dataset Boston Housing (ou similar)
from sklearn.datasets import load_diabetes

diabetes = load_diabetes()
X = diabetes.data[:, :3]  # 3 primeiras features
y = diabetes.target

print('\n=== REGRESSÃO LINEAR MÚLTIPLA ===' )
print(f'\nDataset Diabetes: {X.shape[0]} amostras, {X.shape[1]} features')

# Modelo
model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)

print(f'\nCoeficientes:')
for i, coef in enumerate(model.coef_):
    print(f'  β{i+1}: {coef:.4f}')
print(f'  β₀ (intercept): {model.intercept_:.4f}')

# Métricas
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
mae = mean_absolute_error(y, y_pred)

print(f'\nDesempenho:')
print(f'  R²: {r2:.3f}')
print(f'  RMSE: {rmse:.2f}')
print(f'  MAE: {mae:.2f}')

### O que observar

- Cada coeficiente beta_j tem interpretacao: mudanca em y por unidade de x_j, controlando os demais
- O intercept (beta_0) e a previsao quando TODAS as features sao zero
- R^2 com multiplas features e maior que com uma so (mas pode ser por acaso!)
- RMSE e MAE medem erro em unidades originais de y

### O que concluir

- Mais features nao significa melhor modelo: cada feature adicional pode ser ruido
- Use R^2 ajustado (penaliza pelo numero de features) para comparar modelos com diferentes numeros de variaveis
- Coeficientes em escala original nao sao comparaveis entre si (depende da unidade da feature)
- Para comparar importancia, normalize as features primeiro (StandardScaler)

### Conexao com outros notebooks

- **0_3 (Algebra Linear)**: Multiplicacao de matrizes X*beta e a base do modelo; inversao (X'X)^-1 pode ser instavel
- **1_2 (Inferencial)**: Cada coeficiente pode ser testado com teste t: H0: beta_j = 0 (feature nao contribui)
- **0_8 (Otimizacao)**: Gradient descent e uma alternativa a solucao analitica quando n ou p sao muito grandes

## 3. Diagnostico de Residuos

### Analogia: "O Medico Examina o Paciente Depois do Tratamento"

Ajustar uma regressao e como prescrever um tratamento. O diagnostico de residuos e o exame pos-tratamento: "o tratamento funcionou? Tem efeitos colaterais?"

Os residuos (e_i = y_i - y_hat_i) sao o que o modelo NAO conseguiu explicar. Se o modelo for bom, os residuos devem ser "aleatorios e bem comportados":
1. Media zero (sem vies sistematico)
2. Distribuicao Normal (para validade dos testes t nos coeficientes)
3. Variancia constante (homocedasticidade)
4. Sem padrao temporal (independencia)

### Definicao Formal

Pressupostos OLS (Gauss-Markov):
- E[epsilon] = 0
- Var(epsilon) = sigma^2 * I (homocedasticidade)
- Cov(epsilon_i, epsilon_j) = 0 para i != j (independencia)
- epsilon ~ Normal(0, sigma^2) (para inferencia)

Testes: Shapiro-Wilk (normalidade), Breusch-Pagan (homocedasticidade), Durbin-Watson (autocorrelacao)

### Por que em ML?

Diagnostico de residuos se aplica a QUALQUER modelo, nao so regressao linear. Em ML, analisar residuos revela se o modelo tem vies sistematico (ex: subestima valores altos), se ha heteroscedasticidade (erro maior em certas faixas), ou se ha padroes temporais nao capturados.

In [ ]:
# Resíduos
residuals = y - y_pred

print('\n=== DIAGNÓSTICO DE RESÍDUOS ===' )
print(f'\nPressupostos da Regressão OLS:')
print(f'  1. Linearidade: relação linear entre X e y')
print(f'  2. Normalidade: resíduos ~Normal(0, σ²)')
print(f'  3. Homocedasticidade: variância constante dos resíduos')
print(f'  4. Independência: sem autocorrelação nos resíduos')
print(f'  5. Sem multicolinearidade: features não são colineares')

# 1. Normalidade
from scipy.stats import shapiro, jarque_bera
stat, p = shapiro(residuals)
print(f'\n1. Teste de Normalidade (Shapiro-Wilk):')
print(f'   p-value = {p:.4f}, Normal? {"Sim" if p > 0.05 else "Não"}')

# 2. Homocedasticidade (Breusch-Pagan)
from scipy.stats import chi2

y_pred_std = (y_pred - y_pred.mean()) / y_pred.std()
residuals_sq = residuals**2
X_aux = np.column_stack([np.ones(len(y)), y_pred_std])
model_aux = LinearRegression().fit(X_aux, residuals_sq)
y_aux = model_aux.predict(X_aux)
ss_aux = np.sum((y_aux - residuals_sq.mean())**2)
ss_aux_tot = np.sum((residuals_sq - residuals_sq.mean())**2)
bp_stat = (len(y) / 2) * (ss_aux / ss_aux_tot)
p_bp = 1 - chi2.cdf(bp_stat, df=1)

print(f'\n2. Teste de Homocedasticidade (Breusch-Pagan):')
print(f'   BP statistic = {bp_stat:.4f}, p-value = {p_bp:.4f}')
print(f'   Homocedasticidade? {"Sim" if p_bp > 0.05 else "Não"}')

# Visualização
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Resíduos vs Valores Ajustados
axes[0, 0].scatter(y_pred, residuals, alpha=0.6)
axes[0, 0].axhline(0, color='r', linestyle='--')
axes[0, 0].set_xlabel('Valores Ajustados (ŷ)')
axes[0, 0].set_ylabel('Resíduos')
axes[0, 0].set_title(f'Resíduos vs Ajustados (Homocedasticidade?)')
axes[0, 0].grid(True, alpha=0.3)

# Histograma + KDE
axes[0, 1].hist(residuals, bins=20, alpha=0.7, density=True, edgecolor='black')
mu, sigma = residuals.mean(), residuals.std()
x = np.linspace(residuals.min(), residuals.max(), 100)
axes[0, 1].plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2, label='Normal')
axes[0, 1].set_xlabel('Resíduos')
axes[0, 1].set_ylabel('Densidade')
axes[0, 1].set_title(f'Distribuição de Resíduos (Normal?)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# QQ-plot
stats.probplot(residuals, dist='norm', plot=axes[1, 0])
axes[1, 0].set_title('QQ-plot (Normalidade?)')
axes[1, 0].grid(True, alpha=0.3)

# Scale-Location (√|resíduos padronizados| vs valores ajustados)
residuals_std = residuals / residuals.std()
axes[1, 1].scatter(y_pred, np.sqrt(np.abs(residuals_std)), alpha=0.6)
axes[1, 1].set_xlabel('Valores Ajustados')
axes[1, 1].set_ylabel('√|Resíduos Padronizados|')
axes[1, 1].set_title('Scale-Location (Homocedasticidade?)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### O que observar

- Residuos vs Ajustados: sem padrao visivel = bom; formato de funil = heterocedasticidade
- QQ-plot: pontos na linha diagonal = residuos normais; desvios nas caudas = caudas pesadas
- Histograma: formato de sino = normalidade; assimetria = problema
- Scale-Location: linha horizontal = homocedasticidade; tendencia = variancia nao constante

### O que concluir

- Se residuos NAO sao normais: testes t nos coeficientes perdem validade, mas previsoes podem ainda ser boas
- Se ha heterocedasticidade: use erros padrao robustos (HC0, HC3) ou transforme y (log, sqrt)
- Se ha padrao nos residuos: o modelo esta PERDENDO informacao (adicione features ou use modelo nao-linear)
- Diagnostico de residuos e tao importante quanto o R^2 - um modelo com R^2=0.9 pode ser invalido se residuos violam pressupostos

### Conexao com outros notebooks

- **1_2 (Inferencial)**: Testes de normalidade (Shapiro-Wilk) e exatamente o que fizemos no 1_2
- **1_1 (Descritiva)**: Histogramas e QQ-plots sao ferramentas descritivas aplicadas aos residuos
- **0_6 (Distribuicoes)**: A distribuicao Normal dos residuos conecta com a teoria do 0_6

## 4. Multicolinearidade e VIF

### Analogia: "Dois Sensores Medindo a Mesma Coisa"

Imagine que voce tem dois termometros medindo a mesma temperatura (um em Celsius, outro em Fahrenheit). Se usa ambos como features, o modelo NAO sabe qual "creditar" - oscila entre dar peso a um ou ao outro. O resultado: coeficientes instaveis e erraticos.

VIF (Variance Inflation Factor) mede quanto a variancia do coeficiente INFLA por causa de colinearidade:
- VIF = 1: sem colinearidade
- VIF 1-5: moderada (aceitavel)
- VIF > 5: alta (problema!)
- VIF > 10: severa (acao necessaria)

### Definicao Formal

VIF_j = 1 / (1 - R^2_j), onde R^2_j e o R^2 de regredir x_j contra todas as outras features.

### Por que em ML?

Multicolinearidade nao afeta a PREDICAO do modelo (previsoes sao as mesmas), mas torna os coeficientes ININTERPRETAVEIS e instaveis. Em ML, quando interpretabilidade importa (ex: qual feature mais contribui?), detectar e tratar multicolinearidade e crucial. Ridge regression e a solucao classica.

In [ ]:
# Dados com multicolinearidade
np.random.seed(42)
X_multi = pd.DataFrame({
    'x1': np.random.randn(100),
    'x2': np.random.randn(100),
})

X_multi['x3'] = X_multi['x1'] + 0.01 * np.random.randn(100)  # Colinear com x1
y_multi = 2 * X_multi['x1'] + 3 * X_multi['x2'] + np.random.randn(100) * 2

print('\n=== MULTICOLINEARIDADE ===' )
print(f'\nCorrelação entre features:')
print(X_multi.corr())

# VIF (Variance Inflation Factor)
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_data = pd.DataFrame()
vif_data['Feature'] = X_multi.columns
vif_data['VIF'] = [variance_inflation_factor(X_multi.values, i) for i in range(X_multi.shape[1])]

print(f'\nVariance Inflation Factor (VIF):')
print(vif_data)
print(f'\nInterpretação:')
print(f'  VIF = 1: sem colinearidade')
print(f'  VIF 1-5: colinearidade baixa-moderada')
print(f'  VIF > 5: colinearidade alta (problema!)')

# Impacto na estabilidade dos coeficientes
model_multi = LinearRegression()
model_multi.fit(X_multi, y_multi)

print(f'\nCoeficientes (com multicolinearidade):')
for feat, coef in zip(X_multi.columns, model_multi.coef_):
    print(f'  {feat}: {coef:.4f}')

# Remover x3
X_single = X_multi[['x1', 'x2']]
model_single = LinearRegression()
model_single.fit(X_single, y_multi)

print(f'\nCoeficientes (sem x3 colinear):')
for feat, coef in zip(X_single.columns, model_single.coef_):
    print(f'  {feat}: {coef:.4f}')
print(f'\nNotar: coeficientes mais estáveis quando removemos colinearidade!')


### O que observar

- x1 e x3 sao quase identicas (correlacao ~1.0) - multicolinearidade perfeita
- VIF de x1 e x3 sao enormes (>> 5), enquanto VIF de x2 e proximo de 1
- Com multicolinearidade: coeficiente de x1 e erratico (nao e o esperado beta=2)
- Sem x3 colinear: coeficientes ficam estaveis e proximos dos verdadeiros (2, 3)

### O que concluir

- Multicolinearidade nao muda R^2 nem previsoes, mas torna coeficientes ININTERPRETAVEIS
- Solucoes: remover uma das variaveis colineares, usar Ridge (L2), ou criar componentes principais (PCA)
- Em ML, se so quer previsao, multicolinearidade nao e problema grave; se quer interpretar, e critico
- VIF deve ser calculado DEPOIS de normalizar as features

### Conexao com outros notebooks

- **0_3 (Algebra Linear)**: Multicolinearidade = matriz X'X quase singular = inversao instavel
- **1_1 (Descritiva)**: Matriz de correlacao do 1_1 e o primeiro passo para detectar colinearidade
- **1_3 (Bayesiana)**: Ridge (prior Normal) resolve multicolinearidade adicionando lambda a diagonal de X'X

## 5. Regularizacao: Ridge, Lasso, ElasticNet

### Analogia: "Punir Complexidade para Evitar Memorizacao"

OLS puro encontra os coeficientes que minimizam o erro. Mas com muitas features ou multicolinearidade, os coeficientes podem explodir (valores enormes). Regularizacao adiciona uma PENALIDADE por coeficientes grandes:

- **Ridge (L2)**: Penaliza SOMA DOS QUADRADOS dos coeficientes. Reduz todos, nunca zera.
- **Lasso (L1)**: Penaliza SOMA DOS VALORES ABSOLUTOS. Pode zerar coeficientes (feature selection!).
- **ElasticNet**: Mistura de L1 e L2. O melhor dos dois mundos.

### Definicao Formal

- Ridge: J = SUM(y_i - y_hat_i)^2 + lambda * SUM(beta_j^2)
- Lasso: J = SUM(y_i - y_hat_i)^2 + lambda * SUM(|beta_j|)
- ElasticNet: J = RSS + lambda * (alpha * SUM(|beta_j|) + (1-alpha) * SUM(beta_j^2))

### Por que em ML?

Regularizacao e UBIQUA em ML: weight decay em redes neurais e L2, dropout pode ser visto como L2 aproximado, e sparsity em modelos lineares e L1. A escolha de lambda e a instancia mais simples do dilema vies-variancia: lambda alto = mais vies, menos variancia.

In [ ]:
# Regressão com regularização
alphas = np.logspace(-3, 3, 50)

print('\n=== REGULARIZAÇÃO ===' )
print(f'\nRidge (L2): J = RSS + λΣβⱼ²')
print(f'Lasso (L1): J = RSS + λΣ|βⱼ|')
print(f'ElasticNet: J = RSS + λ(αΣ|βⱼ| + (1-α)Σβⱼ²)')

# Normalizar dados
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_multi)
X_scaled = pd.DataFrame(X_scaled, columns=X_multi.columns)

# Calcular coeficientes para diferentes lambdas
ridge_coefs = []
lasso_coefs = []
elasticnet_coefs = []

for alpha in alphas:
    ridge = Ridge(alpha=alpha).fit(X_scaled, y_multi)
    lasso = Lasso(alpha=alpha, max_iter=10000).fit(X_scaled, y_multi)
    en = ElasticNet(alpha=alpha, l1_ratio=0.5, max_iter=10000).fit(X_scaled, y_multi)
    
    ridge_coefs.append(ridge.coef_)
    lasso_coefs.append(lasso.coef_)
    elasticnet_coefs.append(en.coef_)

ridge_coefs = np.array(ridge_coefs)
lasso_coefs = np.array(lasso_coefs)
elasticnet_coefs = np.array(elasticnet_coefs)

# Visualização
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for i, (coefs, title) in enumerate([(ridge_coefs, 'Ridge (L2)'),
                                      (lasso_coefs, 'Lasso (L1)'),
                                      (elasticnet_coefs, 'ElasticNet')]):
    for j in range(coefs.shape[1]):
        axes[i].plot(np.log10(alphas), coefs[:, j], 'o-', label=f'{X_scaled.columns[j]}')
    axes[i].set_xlabel('log10(λ)')
    axes[i].set_ylabel('Coeficiente')
    axes[i].set_title(f'{title}: Coeficientes vs λ')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nNotar: Lasso reduz coeficientes a zero (feature selection)!')
print(f'       Ridge apenas reduz magnitude.')


### O que observar

- Ridge: coeficientes diminuem gradualmente com lambda, nunca chegam a zero
- Lasso: coeficientes CHEGAM A ZERO a partir de certo lambda (feature selection automatica!)
- ElasticNet: comportamento intermediario entre Ridge e Lasso
- Com lambda grande, todos os modelos convergem para beta = 0 (modelo vazio)

### O que concluir

- Use Ridge quando todas as features provavelmente contribuem (nenhuma e irrelevante)
- Use Lasso quando espera que muitas features sejam irrelevantes (sparse model)
- Use ElasticNet quando tem muitas features correlacionadas (Lasso pode ser instavel nesse caso)
- Lambda e um HIPERPARAMETRO que deve ser escolhido por validacao cruzada (proxima secao)

### Conexao com outros notebooks

- **1_3 (Bayesiana)**: Ridge = MAP com prior Normal, Lasso = MAP com prior Laplace
- **0_8 (Otimizacao)**: Regularizacao modifica a funcao de custo que o otimizador minimiza
- **0_4 (Calculo)**: A derivada da penalidade L2 e 2*lambda*beta; a derivada de L1 e lambda*sign(beta)

## 6. Selecao de Modelos com Validacao Cruzada

### Analogia: "Testar o Aluno com Provas que Ele Nunca Viu"

Validacao cruzada simula o que aconteceria com dados novos: divide os dados em K partes, treina em K-1 e testa na restante, repete K vezes. A media dos erros e uma estimativa honesta da performance.

Para escolher lambda (regularizacao), faca CV para cada valor de lambda e escolha o que minimiza o erro de validacao.

### Definicao Formal

K-Fold CV: para cada fold k=1,...,K, treina no complemento e avalia no fold k. Metrica = media dos K erros.

### Por que em ML?

Validacao cruzada e O metodo padrao para selecao de hiperparametros em ML. GridSearchCV e RandomizedSearchCV do sklearn automatizam isso. A regra "treinar em dados diferentes dos que avalia" e o principio mais importante de ML.

In [ ]:
# Validação cruzada
from sklearn.model_selection import cross_validate

print('\n=== SELEÇÃO DE MODELOS ===' )

alphas_test = np.logspace(-2, 2, 20)
ridges = [Ridge(alpha=a) for a in alphas_test]
lassos = [Lasso(alpha=a, max_iter=10000) for a in alphas_test]

# Cross-validation
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

ridge_mse = []
lasso_mse = []

for model in ridges:
    scores = cross_val_score(model, X_scaled, y_multi, cv=kfold, scoring='neg_mean_squared_error')
    ridge_mse.append(-scores.mean())

for model in lassos:
    scores = cross_val_score(model, X_scaled, y_multi, cv=kfold, scoring='neg_mean_squared_error')
    lasso_mse.append(-scores.mean())

best_ridge_idx = np.argmin(ridge_mse)
best_lasso_idx = np.argmin(lasso_mse)

print(f'\nMelhor λ (Ridge): {alphas_test[best_ridge_idx]:.4f}, MSE = {ridge_mse[best_ridge_idx]:.2f}')
print(f'Melhor λ (Lasso): {alphas_test[best_lasso_idx]:.4f}, MSE = {lasso_mse[best_lasso_idx]:.2f}')

# Visualização
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(np.log10(alphas_test), ridge_mse, 'o-', linewidth=2, label='Ridge (L2)')
ax.plot(np.log10(alphas_test), lasso_mse, 's-', linewidth=2, label='Lasso (L1)')
ax.axvline(np.log10(alphas_test[best_ridge_idx]), color='blue', linestyle='--', alpha=0.5)
ax.axvline(np.log10(alphas_test[best_lasso_idx]), color='orange', linestyle='--', alpha=0.5)
ax.set_xlabel('log10(λ)')
ax.set_ylabel('MSE (5-fold CV)')
ax.set_title('Seleção de λ por Validação Cruzada')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### O que observar

- A curva de MSE vs lambda tem formato de U (muito baixo = overfitting, muito alto = underfitting)
- O lambda otimo esta no "fundo" da curva U
- Ridge e Lasso podem ter lambdas otimos diferentes para o mesmo dataset
- A escala log e essencial para visualizar (lambdas variam em ordens de magnitude)

### O que concluir

- NUNCA escolha lambda manualmente - use CV sistematicamente
- O lambda otimo depende do dataset; nao existe valor "universal"
- K=5 ou K=10 sao as escolhas mais comuns para o numero de folds
- Se o dataset e muito pequeno, use Leave-One-Out CV (K=n)
- Em sklearn, RidgeCV e LassoCV fazem tudo automaticamente

### Conexao com outros notebooks

- **1_2 (Inferencial)**: CV e uma alternativa a AIC/BIC para selecao de modelo; mais robusto com amostras finitas
- **1_5 (Design de Experimentos)**: A divisao em folds e uma forma de design experimental dentro do ML
- **0_8 (Otimizacao)**: A busca por lambda otimo e um problema de otimizacao de hiperparametros

## 7. Regressao Polinomial e Overfitting

### Analogia: "Ajustar uma Curva vs Memorizar os Dados"

Se a relacao entre x e y nao e linear, podemos adicionar termos polinomiais: x^2, x^3, etc. O modelo continua "linear" nos parametros (beta_0 + beta_1*x + beta_2*x^2 + ...), apenas as features sao nao-lineares.

O perigo: um polinomio de grau n-1 passa EXATAMENTE por n pontos (R^2 treino = 1.0), mas pode ser completamente errado entre os pontos. Isso e OVERFITTING.

### Por que em ML?

Regressao polinomial e o exemplo mais didatico do dilema vies-variancia: grau baixo = alto vies (underfitting), grau alto = alta variancia (overfitting). A escolha do grau e analoga a escolha da complexidade em qualquer modelo de ML.

In [ ]:
# Dados com padrão não-linear
np.random.seed(42)
X_poly = np.linspace(-3, 3, 50).reshape(-1, 1)
y_poly = np.sin(X_poly).ravel() + np.random.randn(50) * 0.2

print('\n=== REGRESSÃO POLINOMIAL ===' )
print(f'\nDados: Padrão não-linear (seno + ruído)')

# Diferentes graus de polinômio
degrees = [1, 3, 5, 9]
poly_models = []

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

X_test = np.linspace(-3, 3, 200).reshape(-1, 1)

for idx, degree in enumerate(degrees):
    poly = PolynomialFeatures(degree=degree)
    X_poly_feat = poly.fit_transform(X_poly)
    X_test_feat = poly.transform(X_test)
    
    model = LinearRegression().fit(X_poly_feat, y_poly)
    y_pred = model.predict(X_poly_feat)
    y_test_pred = model.predict(X_test_feat)
    
    r2_train = r2_score(y_poly, y_pred)
    r2_test = r2_score(np.sin(X_test).ravel(), y_test_pred)
    
    axes[idx].scatter(X_poly, y_poly, alpha=0.6, label='Dados')
    axes[idx].plot(X_test, y_test_pred, 'r-', linewidth=2, label='Ajuste')
    axes[idx].set_xlabel('X')
    axes[idx].set_ylabel('y')
    axes[idx].set_title(f'Grau {degree}: R² Train={r2_train:.3f}, Test={r2_test:.3f}')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)
    axes[idx].set_ylim(-1.5, 1.5)

plt.tight_layout()
plt.show()

print(f'Notar: Grau 1 = underfitting, Grau 9 = overfitting!')
print(f'       Graus 3-5 parecem balanceados.')


### O que observar

- Grau 1: linha reta nao captura o padrao senoidal (underfitting, R^2 treino baixo)
- Grau 3: boa aproximacao do padrao (bom equilibrio)
- Grau 5: ajuste bom mas comeca a oscilar nas bordas
- Grau 9: ajuste perfeito ao treino mas previsoes loucas fora do range (overfitting)

### O que concluir

- O R^2 de TREINO SEMPRE aumenta com mais complexidade; e o R^2 de TESTE que importa
- O grau otimo e aquele que maximiza R^2 de teste (ou minimiza erro de CV)
- Regularizacao pode "salvar" um modelo polinomial de grau alto (reduz coeficientes extremos)
- Na pratica, prefira modelos mais simples que performam similarmente (principio de parcimonia)

### Conexao com outros notebooks

- **0_8 (Otimizacao)**: Overfitting em polinomios e o mesmo fenomeno que em redes neurais (memorizar treino)
- **1_5 (Design de Experimentos)**: A divisao treino/teste e essencial para detectar overfitting
- **0_4 (Calculo)**: Polinomios de Taylor sao a base teorica da aproximacao polinomial

## 8. Learning Curves: Diagnostico de Vies/Variancia

### Analogia: "Quao Rapido Meu Modelo Aprende?"

Learning curves mostram como o erro de treino e validacao mudam conforme o tamanho do dataset:
- **Alto vies**: Ambas as curvas convergem para erro ALTO. Mais dados nao ajudam.
- **Alta variancia**: Gap entre treino (erro baixo) e validacao (erro alto). Mais dados podem ajudar.
- **Bom ajuste**: Ambas convergem para erro BAIXO com gap pequeno.

### Por que em ML?

Learning curves sao diagnostico essencial: dizem se voce precisa de MAIS DADOS, MAIS FEATURES, ou MODELO DIFERENTE. Evitam desperdicar recursos coletando dados quando o problema e modelo errado.

In [ ]:
# Learning curves
from sklearn.model_selection import learning_curve

print('\n=== LEARNING CURVES ===' )

train_sizes = np.linspace(0.1, 1.0, 10)
train_sizes_abs, train_scores, val_scores = learning_curve(
    LinearRegression(),
    X_scaled, y_multi,
    train_sizes=train_sizes,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

train_scores = -train_scores
val_scores = -val_scores

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(train_sizes_abs, train_mean, 'o-', linewidth=2, color='blue', label='Training score')
ax.fill_between(train_sizes_abs, train_mean - train_std, train_mean + train_std, alpha=0.2, color='blue')
ax.plot(train_sizes_abs, val_mean, 'o-', linewidth=2, color='red', label='Validation score')
ax.fill_between(train_sizes_abs, val_mean - val_std, val_mean + val_std, alpha=0.2, color='red')

ax.set_xlabel('Tamanho do Treino')
ax.set_ylabel('MSE')
ax.set_title('Learning Curve (Linear Regression)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nInterpretação:')
print(f'  - Gap grande entre treino e validação: overfitting/alta variância')
print(f'  - Ambas as curvas altas: underfitting/alto viés')
print(f'  - Curvas convergentes e baixas: bom ajuste')


### O que observar

- Erro de treino comeca em zero (com 1 ponto, ajuste perfeito) e SOBE conforme n cresce
- Erro de validacao comeca alto (pouco dado) e DESCE conforme n cresce
- As duas curvas convergem para um valor similar = modelo esta funcionando
- O gap entre as curvas e a "variancia" do modelo

### O que concluir

- Se as curvas ja convergiram: mais dados NAO ajudarao; tente modelo mais complexo ou melhores features
- Se ha gap grande: mais dados devem ajudar; ou regularize mais para reduzir variancia
- Se ambas sao altas: modelo muito simples (underfitting); aumente complexidade
- Learning curves sao mais informativas que uma unica metrica de CV

### Conexao com outros notebooks

- **0_8 (Otimizacao)**: Learning rate curves em gradient descent sao analogas a learning curves de modelo
- **1_2 (Inferencial)**: O gap treino-validacao e uma medida do "erro tipo I" do modelo (falsa confianca)
- **1_5 (Design de Experimentos)**: Tamanho amostral necessario conecta com a curva de aprendizado

## 9. Exercicios Praticos

### Exercicio 1: Regressao Completa com Diagnostico

Usando o dataset Diabetes do sklearn:
1. Ajuste regressao linear com as 3 primeiras features
2. Calcule R^2, RMSE e MAE
3. Faca diagnostico de residuos (normalidade, homocedasticidade)
4. Compare com Ridge (lambda=1.0)

In [ ]:
# TAREFA DO ALUNO: Exercicio 1 - Regressao com Diagnostico
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression, Ridge

diabetes = load_diabetes()
X = diabetes.data[:, :3]
y = diabetes.target

# 1. Ajustar regressao linear
# model_ols = None  # TAREFA DO ALUNO
# y_pred_ols = None  # TAREFA DO ALUNO

# 2. Metricas
# r2_ols = None  # TAREFA DO ALUNO
# rmse_ols = None  # TAREFA DO ALUNO
# mae_ols = None  # TAREFA DO ALUNO

# 3. Diagnostico de residuos
# residuals = None  # TAREFA DO ALUNO
# Shapiro-Wilk test
# stat_norm, p_norm = None  # TAREFA DO ALUNO

# 4. Ridge com lambda=1.0
# model_ridge = None  # TAREFA DO ALUNO
# r2_ridge = None  # TAREFA DO ALUNO

# Compare e imprima resultados

In [ ]:
# SOLUCAO - Exercicio 1
from sklearn.datasets import load_diabetes

diabetes = load_diabetes()
X = diabetes.data[:, :3]
y = diabetes.target

print('=== EXERCICIO 1: REGRESSAO COM DIAGNOSTICO ===')

# 1. OLS
model_ols = LinearRegression().fit(X, y)
y_pred_ols = model_ols.predict(X)

# 2. Metricas
r2_ols = r2_score(y, y_pred_ols)
rmse_ols = np.sqrt(mean_squared_error(y, y_pred_ols))
mae_ols = mean_absolute_error(y, y_pred_ols)

print(f'\n1-2. OLS:')
print(f'   Coeficientes: {model_ols.coef_}')
print(f'   R^2 = {r2_ols:.3f}')
print(f'   RMSE = {rmse_ols:.2f}')
print(f'   MAE = {mae_ols:.2f}')

# 3. Diagnostico
residuals = y - y_pred_ols
stat_norm, p_norm = stats.shapiro(residuals)
print(f'\n3. Diagnostico:')
print(f'   Shapiro-Wilk: p = {p_norm:.4f} -> {"Normal" if p_norm > 0.05 else "Nao Normal"}')
print(f'   Media residuos: {residuals.mean():.4f} (deveria ser ~0)')
print(f'   Std residuos: {residuals.std():.2f}')

# 4. Ridge
model_ridge = Ridge(alpha=1.0).fit(X, y)
y_pred_ridge = model_ridge.predict(X)
r2_ridge = r2_score(y, y_pred_ridge)

print(f'\n4. Comparacao OLS vs Ridge:')
print(f'   R^2 OLS:   {r2_ols:.3f}')
print(f'   R^2 Ridge: {r2_ridge:.3f}')
print(f'   Coef OLS:   {model_ols.coef_}')
print(f'   Coef Ridge: {model_ridge.coef_}')
print(f'   Ridge tem coeficientes menores (regularizados)')

### Exercicio 2: Regularizacao e Feature Selection

1. Crie dados com 10 features, sendo 3 reais e 7 ruido
2. Ajuste Lasso com diferentes lambdas
3. Identifique quais features o Lasso "seleciona" (coeficiente != 0)
4. Compare com os verdadeiros coeficientes

In [ ]:
# TAREFA DO ALUNO: Exercicio 2 - Feature Selection com Lasso
np.random.seed(42)
n = 200
X_ex2 = np.random.randn(n, 10)  # 10 features
# Apenas features 0, 1, 2 sao relevantes (coefs: 3, -2, 1.5)
y_ex2 = 3*X_ex2[:, 0] - 2*X_ex2[:, 1] + 1.5*X_ex2[:, 2] + np.random.randn(n)*0.5

# 1. Normalizar
# scaler = StandardScaler()
# X_scaled = None  # TAREFA DO ALUNO

# 2. Lasso com diferentes lambdas
# lambdas = [0.01, 0.1, 0.5, 1.0, 5.0]
# for lam in lambdas:
#     model = None  # TAREFA DO ALUNO: Lasso(alpha=lam)
#     coefs = None  # TAREFA DO ALUNO
#     n_nonzero = None  # TAREFA DO ALUNO: contar coeficientes != 0

# Imprima resultado

In [ ]:
# SOLUCAO - Exercicio 2
np.random.seed(42)
n = 200
X_ex2 = np.random.randn(n, 10)
y_ex2 = 3*X_ex2[:, 0] - 2*X_ex2[:, 1] + 1.5*X_ex2[:, 2] + np.random.randn(n)*0.5

print('=== EXERCICIO 2: FEATURE SELECTION COM LASSO ===')
print(f'Coeficientes verdadeiros: [3, -2, 1.5, 0, 0, 0, 0, 0, 0, 0]')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_ex2)

lambdas = [0.01, 0.1, 0.5, 1.0, 5.0]
print(f'\nLasso com diferentes lambdas:')
for lam in lambdas:
    model = Lasso(alpha=lam, max_iter=10000).fit(X_scaled, y_ex2)
    coefs = model.coef_
    n_nonzero = np.sum(np.abs(coefs) > 0.001)
    print(f'  lambda={lam:5.2f}: {n_nonzero} features ativas, coefs: {np.round(coefs, 2)}')

print(f'\nConclusao: Lasso com lambda adequado (~0.1-0.5) seleciona')
print(f'as 3 features corretas e zera as 7 irrelevantes')

### Exercicio 3: Polynomial Degree Selection

1. Gere dados senoidais com ruido
2. Ajuste polinomios de grau 1, 3, 5, 7, 15
3. Calcule R^2 de treino E de teste (70/30 split)
4. Identifique o grau otimo

In [ ]:
# TAREFA DO ALUNO: Exercicio 3 - Polynomial Degree Selection
np.random.seed(42)
X_ex3 = np.linspace(-3, 3, 80).reshape(-1, 1)
y_ex3 = np.sin(X_ex3).ravel() + np.random.randn(80) * 0.2

# 1. Train/test split
# X_train, X_test, y_train, y_test = None  # TAREFA DO ALUNO

# 2-3. Para cada grau, ajuste e calcule R^2 treino e teste
# degrees = [1, 3, 5, 7, 15]
# for degree in degrees:
#     poly = PolynomialFeatures(degree=degree)
#     X_train_poly = None  # TAREFA DO ALUNO
#     X_test_poly = None  # TAREFA DO ALUNO
#     model = None  # TAREFA DO ALUNO
#     r2_train = None  # TAREFA DO ALUNO
#     r2_test = None  # TAREFA DO ALUNO

# 4. Identifique grau otimo

In [ ]:
# SOLUCAO - Exercicio 3
np.random.seed(42)
X_ex3 = np.linspace(-3, 3, 80).reshape(-1, 1)
y_ex3 = np.sin(X_ex3).ravel() + np.random.randn(80) * 0.2

print('=== EXERCICIO 3: POLYNOMIAL DEGREE SELECTION ===')

X_train, X_test, y_train, y_test = train_test_split(X_ex3, y_ex3, test_size=0.3, random_state=42)

degrees = [1, 3, 5, 7, 15]
results = []

print(f'\nGrau | R^2 Treino | R^2 Teste | Status')
print(f'-----|-----------|----------|-------')
for degree in degrees:
    poly = PolynomialFeatures(degree=degree)
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly = poly.transform(X_test)

    model = LinearRegression().fit(X_train_poly, y_train)
    r2_train = r2_score(y_train, model.predict(X_train_poly))
    r2_test = r2_score(y_test, model.predict(X_test_poly))

    if r2_test < 0:
        status = 'OVERFITTING SEVERO'
    elif r2_train - r2_test > 0.1:
        status = 'Overfitting'
    elif r2_test < 0.5:
        status = 'Underfitting'
    else:
        status = 'BOM'

    results.append((degree, r2_train, r2_test))
    print(f'  {degree:2d}  |   {r2_train:.3f}   |  {r2_test:.3f}  | {status}')

best = max(results, key=lambda x: x[2])
print(f'\nGrau otimo: {best[0]} (R^2 teste = {best[2]:.3f})')

### O que observar nos exercicios

- Exercicio 1 mostra o fluxo completo: treinar -> diagnosticar -> iterar. Este ciclo e a rotina real de qualquer modelagem
- Exercicio 2 revela que Lasso zera coeficientes irrelevantes, funcionando como selecao automatica de features
- Exercicio 3 demonstra na pratica o tradeoff vies-variancia: grau baixo = underfitting, grau alto = overfitting

### O que concluir dos exercicios

- **Diagnostico nao e opcional**: sem ele, voce nao sabe se o modelo e confiavel
- **Regularizacao e feature selection estao conectados**: Lasso faz ambos simultaneamente
- **Validacao cruzada e o arbitro final**: ela evita que o grau polinomial seja escolhido por overfitting no treino

### Conexao com outros notebooks

- O ciclo treinar-diagnosticar-iterar dos exercicios antecipa o pipeline completo de `4_1_pipeline_ml`
- A selecao de features do Exercicio 2 conecta com `3_1_feature_engineering`

### O que observar no panorama geral

- A regressao e o *unico* modelo que permite interpretar diretamente o efeito de cada variavel (coeficientes)
- Todos os diagnosticos (residuos, VIF, learning curves) sao ferramentas de *validacao*, nao de *treino*

### O que concluir do panorama geral

- **Regressao e a base de quase todo ML supervisionado**: redes neurais sao regressoes empilhadas com ativacoes nao-lineares
- **O ciclo modelar -> diagnosticar -> melhorar e universal**: vale para regressao, arvores, ensembles, deep learning

### Conexao com outros notebooks

- A interpretabilidade dos coeficientes conecta com `5_3_interpretabilidade_modelos`
- O conceito de funcao de perda (RSS) reaparece em `2_3_funcoes_de_perda` com generalizacoes

## 10. Erros Comuns e Armadilhas

### Erro 1: Confiar so em R^2

**Errado**: "R^2 = 0.95, o modelo e otimo!"
**Correto**: R^2 mede ajuste no TREINO; pode ser overfitting. Sempre valide com dados separados (CV). R^2 NUNCA diminui ao adicionar variaveis (use R^2 ajustado).

### Erro 2: Interpretar coeficientes em escala original

**Errado**: "beta_peso = 5 e beta_idade = 0.1, logo peso e mais importante"
**Correto**: Se peso esta em kg e idade em anos, os coeficientes nao sao comparaveis. Normalize (StandardScaler) antes de comparar importancia relativa.

### Erro 3: Ignorar multicolinearidade

**Errado**: "Vou usar todas as features e deixar o modelo decidir"
**Correto**: Com features correlacionadas, coeficientes sao instaveis. Calcule VIF; se > 5, considere remover ou usar Ridge.

### Erro 4: Violar pressupostos sem perceber

**Errado**: "Ajustei o modelo, fiz previsao, acabei"
**Correto**: Sempre diagnostique residuos (QQ-plot, residuos vs ajustados). Se pressupostos sao violados, testes t nos coeficientes sao invalidos.

### Erro 5: Usar polinomio de grau alto sem regularizacao

**Errado**: "Grau 15 tem R^2 treino perfeito!"
**Correto**: Polinomio alto memoriza ruido. Use CV para escolher grau ou adicione regularizacao.

### Erro 6: Esquecer que correlacao e uma RETA

**Errado**: "Correlacao = 0, logo nao ha relacao"
**Correto**: Um padrao em U tem correlacao ~0 mas relacao forte! Sempre visualize antes de concluir.

### Erro 7: Extrapolar alem do range dos dados

**Errado**: "O modelo diz que uma pessoa de 300cm pesaria 500kg"
**Correto**: Regressao so e valida no RANGE dos dados de treino. Extrapolacao e pura especulacao.

## 11. Resumo e Conexoes

### Hierarquia dos Conceitos

```
DADOS (X, y)
    |
    v
OLS (Ordinary Least Squares)
    |
    |---> Encontra beta que minimiza SUM(y - y_hat)^2
    |---> Solucao analitica: beta = (X'X)^-1 X'y
    |---> Metricas: R^2, RMSE, MAE
    |
    v
DIAGNOSTICO DE PRESSUPOSTOS
    |---> Linearidade: scatter + residuos vs ajustados
    |---> Normalidade: QQ-plot, Shapiro-Wilk
    |---> Homocedasticidade: Scale-Location, Breusch-Pagan
    |---> Multicolinearidade: VIF < 5
    |
    v
SE PROBLEMAS ENCONTRADOS
    |---> Nao-linearidade -> polynomial features ou transformacoes
    |---> Multicolinearidade -> Ridge (L2) ou remover features
    |---> Muitas features irrelevantes -> Lasso (L1, feature selection)
    |---> Ambos -> ElasticNet (L1 + L2)
    |
    v
VALIDACAO E SELECAO
    |---> K-fold CV: escolhe hiperparametros (lambda, grau)
    |---> Learning curves: diagnostica vies/variancia
    |---> Principio: treinar != avaliar (dados separados!)
```

### Tabela de Conexoes

| Conceito deste notebook | Conecta com | Como |
|------------------------|-------------|------|
| OLS (minimos quadrados) | 0_4 (Calculo) | Derivar MSE e igualar a zero = gradiente |
| Solucao matricial | 0_3 (Algebra) | beta = (X'X)^-1 X'y usa inversao de matrizes |
| Testes t nos coeficientes | 1_2 (Inferencial) | H0: beta_j = 0 e um teste t padrao |
| Regularizacao = MAP | 1_3 (Bayesiana) | Ridge = prior Normal, Lasso = prior Laplace |
| R^2 e correlacao | 1_1 (Descritiva) | r^2 de Pearson = R^2 em regressao simples |
| Gradient descent | 0_8 (Otimizacao) | Alternativa a solucao analitica para n grande |
| Overfitting/underfitting | 1_5 (Design) | CV e learning curves para detectar |
| Normalidade residuos | 0_6 (Distribuicoes) | Pressupostos da Normal se aplicam aqui |

### Checklist de Competencias

- [ ] Sei ajustar regressao linear simples e multipla e interpretar coeficientes
- [ ] Sei calcular e interpretar R^2, RMSE, MAE
- [ ] Sei diagnosticar residuos (normalidade, homocedasticidade, linearidade)
- [ ] Sei detectar multicolinearidade com VIF e trata-la
- [ ] Sei usar Ridge, Lasso e ElasticNet e escolher entre eles
- [ ] Sei selecionar lambda por validacao cruzada
- [ ] Sei identificar overfitting com regressao polinomial
- [ ] Sei usar learning curves para diagnosticar vies/variancia

### Proximos Passos

No notebook **1_5 (Design de Experimentos)**, voce aprendera a PLANEJAR a coleta de dados e a validacao de modelos de forma rigorosa. A divisao treino/teste, a validacao cruzada e o calculo de tamanho amostral que usamos aqui serao formalizados como parte de um framework de design experimental.